Installation

In [2]:
!pip install -q -U openai-whisper
!pip install -q -U unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 19.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 121.5 MB/s eta 0:

Imports

In [3]:
import torch
import whisper
import unsloth

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Loading Whisper

In [4]:
whisper_model = whisper.load_model("base")

100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 101MiB/s]


Audio File

In [21]:
audio_path = "/content/Question1.mp3"

Loading Qwen

In [14]:
from unsloth import FastLanguageModel

max_seq_length = 2048

model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"

reasoning_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Prepare Inference

In [15]:
FastLanguageModel.for_inference(reasoning_model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048, padding_idx=151654)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen

Getting GPU Memory

In [16]:
import time
import torch

def get_gpu_memory():
    if not torch.cuda.is_available():
        return 0.0

    return torch.cuda.memory_allocated() / (1024 ** 3)

Transcribing Audio

In [17]:
def transcribe_audio(audio_path):
    result = whisper_model.transcribe(
        audio_path,
        fp16 = torch.cuda.is_available()
    )

    return result["text"].strip()

Generating Reasoning Response

In [18]:
def generate_reasoning_response(text):
    """
    Generate an answer using the 4-bit quantized Qwen model.
    """

    prompt = f"""
You are a helpful reasoning assistant.

A user asked the following question through an audio recording.
The question was transcribed using Whisper.

Transcribed question:
{text}

Provide a clear, accurate, and logically reasoned answer.
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.inference_mode():

        outputs = reasoning_model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )

    input_length = inputs["input_ids"].shape[1]

    generated_tokens = outputs[0][input_length:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return answer

In [19]:
def run_audio_qa(audio_path):
    """
    Complete Audio → Whisper → Qwen pipelin with timing and GPU memory measurements.
    """

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    pipeline_start = time.time()

    memory_before = get_gpu_memory()

    whisper_start = time.time()

    transcription = transcribe_audio(audio_path)

    whisper_time = time.time() - whisper_start

    reasoning_start = time.time()

    answer = generate_reasoning_response(
        transcription
    )

    reasoning_time = time.time() - reasoning_start

    total_time = time.time() - pipeline_start

    memory_after = get_gpu_memory()

    if torch.cuda.is_available():
        peak_memory = (
            torch.cuda.max_memory_allocated()
            / (1024 ** 3)
        )
    else:
        peak_memory = 0.0

    return {
        "transcription": transcription,
        "answer": answer,
        "whisper_time": whisper_time,
        "reasoning_time": reasoning_time,
        "total_time": total_time,
        "memory_before": memory_before,
        "memory_after": memory_after,
        "peak_memory": peak_memory
    }

In [22]:
result = run_audio_qa(audio_path)

print("\nTRANSCRIPTION:\n")
print(result["transcription"])

print("\nANSWER:\n")
print(result["answer"])

print("\nPERFORMANCE\n")

print(f"Whisper time: {result['whisper_time']:.2f} seconds")
print(f"Qwen reasoning time: {result['reasoning_time']:.2f} seconds")
print(f"Total pipeline time: {result['total_time']:.2f} seconds")
print(f"GPU memory before: {result['memory_before']:.2f} GB")
print(f"GPU memory after: {result['memory_after']:.2f} GB")
print(f"Peak GPU memory: {result['peak_memory']:.2f} GB")

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TRANSCRIPTION:

What are the three major subrimeons in bioinformatics?

ANSWER:

In bioinformatics, the term "subrimeon" is not a standard or recognized concept. It's possible that there might be a typo or confusion with other terms. The field of bioinformatics primarily deals with the application of computational techniques to analyze biological data. If you could clarify the term or provide more context, I could offer a more precise answer. However, based on common bioinformatics topics, some key areas include:

1. Sequence Analysis: This involves analyzing DNA, RNA, and protein sequences to understand their structure, function, and evolutionary relationships.
2. Structural Bioinformatics: This focuses on the analysis of 3D structures of biomolecules, often using tools like molecular docking and virtual screening.
ml
The transcription provided by Whisper seems to have a typo or misinterpretation. In bioinformatics, the term "subrimeon" does not exist. However, if we consider a relat